# 📩 Email Data Preprocessing and Message Extraction

This notebook processes a set of parsed email data from a historical institutional archive (over 20 years old). The emails were shared in a structured format as `.eml` files. The goal is to retain original metadata (file paths, folder structure, and filenames), then extract and clean individual messages from email chains for further analysis and archiving.

> **Note:** All personally identifiable information (PII) has been removed from this notebook. The sample outputs shown use anonymized placeholder values. Raw email files are not included in this repository.


## 🔹 Step 1: Load the Parsed Email Data

We start by loading the previously parsed email metadata and content stored in a CSV file.

In [ ]:
import os
import pandas as pd

df = pd.read_csv(r"data/emails_parsed_cleaned_PII_Extracted_v3.csv")
df.head()

## 🔹 Step 2: Extract Folder Structure from File Path

Each email file path contains useful metadata (such as the folder it was in and the original file name). We split the file path on backslashes `\\` to break this down into separate columns.

In [ ]:
# 1) Split the 'file_path' column on backslashes
split_path_df = df["file_path"].str.split(r'\\', expand=True)

# 2) Rename the columns in the new DataFrame
split_path_df.columns = [f"Folder_{i+1}" for i in range(split_path_df.shape[1])]

# 3) Join the split path columns back to the original DataFrame
df = df.join(split_path_df)

df.head()

## 🔹 Step 3: Extract Email Title (File Name without Extension)

We extract the original email file name (without the `.eml` extension) and store it in a new column called `email_title`. This helps track message context even after separating multi-part chains.

In [ ]:
def get_email_title(path):
    if not path:
        return ""
    base_name = os.path.basename(path)
    title_without_ext, _ = os.path.splitext(base_name)
    return title_without_ext

df["email_title"] = df["file_path"].apply(get_email_title)
df = df.reset_index()
df.head()

## 📬 Step 4: Break Email Chains into Individual Messages

Many email files are long threads with multiple forwarded or replied messages inside. To preserve each part of the conversation, we split messages based on the delimiter:

```
-----Original Message-----
```

This delimiter usually marks the start of a new message within an email thread.


In [ ]:
import pandas as pd

# 1. Split on the chain delimiter to get individual messages as a list
df["email_original_message"] = df["email_text"].str.split("-----Original Message-----")

# 2. Explode so each message becomes its own row
df_exploded = df.explode("email_original_message").reset_index(drop=True)

# 3. Strip leading/trailing whitespace
df_exploded["email_original_message"] = df_exploded["email_original_message"].str.strip()

df_exploded.head()

In [ ]:
# Shape of exploded dataframe
print(f"Original rows: {len(df)}")
print(f"Exploded rows (one per message segment): {len(df_exploded)}")

Original rows: 5810
Exploded rows (one per message segment): 79676


## 🗂️ Step 5: Extracting Email Header Information

This section extracts structured metadata (like sender, recipient, subject, etc.) from each individual email message using regular expressions.

**Steps:**
1. **Import Libraries** — `re` and `pandas` for regex parsing and DataFrame handling.
2. **Define `extract_headers()` Function** — Parses `From`, `Sent`, `To`, `Cc` (optional), `Subject`, and `Importance` (optional) from the header block. The remainder becomes `email_text_content`.
3. **Apply to Messages** — Applied row-by-row to `email_original_message`.
4. **Convert to DataFrame** — Header dicts become new columns.
5. **Merge with Original Data** — Concatenated back into `df_exploded`.
6. **Save Final Output** — Written to CSV for downstream use.


In [ ]:
import re
import pandas as pd

def extract_headers(message):
    if not isinstance(message, str):
        message = ""
    
    header_pattern = (
        r"^From:\s*(?P<from_email>.*?)\r?\n"
        r"Sent:\s*(?P<sent_email>.*?)\r?\n"
        r"To:\s*(?P<to_email>.*?)\r?\n"
        r"(?:Cc:\s*(?P<cc_email>.*?)\r?\n)?"
        r"Subject:\s*(?P<subject_email>.*?)\r?\n"
        r"(?:Importance:\s*(?P<importance_email>.*?)\r?\n)?"
        r"\r?\n"
    )
    m = re.search(header_pattern, message, re.DOTALL)
    if m:
        headers = m.groupdict()
        email_text_content = message[m.end():].lstrip()
        headers["email_text_content"] = email_text_content
        return headers
    else:
        return {
            "from_email": None,
            "sent_email": None,
            "to_email": None,
            "cc_email": None,
            "subject_email": None,
            "importance_email": None,
            "email_text_content": message
        }

# Apply header extraction
header_data = df_exploded["email_original_message"].apply(extract_headers)
header_df = header_data.apply(pd.Series)
df_exploded = pd.concat([df_exploded, header_df], axis=1)

# Save
# df_exploded.to_csv(r"data/emails_parsed_exploded.csv")
print(df_exploded.shape)

## 🧾 Step 6: Viewing Extracted Email Data

This section demonstrates how to view both the original email message and the structured header fields we extracted. We verify that header extraction is working correctly.

In [ ]:
# Load exploded data
df_exploded = pd.read_csv(r"data/emails_parsed_exploded.csv", low_memory=False)
df_exploded = df_exploded.drop(columns=['Unnamed: 0', 'level_0', 'index'], errors='ignore')
print(df_exploded.shape)

In [ ]:
# Print a sample extracted record (row 18 in original notebook)
# Shown here with anonymized placeholder values matching the PII extraction format
sample_row = 18

print("Original Email:\n")
print(df_exploded["email_original_message"][sample_row])
print("\n")
print("Extracted Header Contents:\n")
print("From: " + str(df_exploded["from_email"][sample_row]))
print("Sent: " + str(df_exploded["sent_email"][sample_row]))
print("To: " + str(df_exploded["to_email"][sample_row]))
print("Cc: " + str(df_exploded["cc_email"][sample_row]))
print("Subject: " + str(df_exploded["subject_email"][sample_row]))
print("Importance: " + str(df_exploded["importance_email"][sample_row]))

**Example output (anonymized):**

```
Original Email:

From:     [SENDER NAME REDACTED]
Sent:     Wednesday, [DATE REDACTED]
To:       [RECIPIENT(S) REDACTED]
Cc:       [CC REDACTED]
Subject:  RE: Network access policy update
Importance: High

I want to discourage shared group logins. Eventually they will be disabled
as there is no personal accountability for network activity.

Extracted Header Contents:

From:       [SENDER NAME REDACTED]
Sent:       Wednesday, [DATE REDACTED]
To:         [RECIPIENT(S) REDACTED]
Cc:         [CC REDACTED]
Subject:    RE: Network access policy update
Importance: High
```


## 💡 Ideas for Extracted Features

---

## 🔍 Step 7: Searching for Emails Sent by a Specific Person

We can now search and filter emails based on the sender using fuzzy matching. This allows us to identify messages sent by a specific person, even if the name is slightly misspelled or formatted differently.

This approach ensures we catch variations like `"A. Contact <contact@domain.com>"`, `"Contact, A."`, or similar. This is especially useful in older email archives where formatting may be inconsistent.


In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Set the name you're trying to match (use your target sender name here)
target_name = "Sender, Example"

# Set a similarity threshold (80 or higher recommended)
threshold = 80

# Create a boolean mask based on fuzzy matching
matches = df_exploded["from_email"].apply(
    lambda x: fuzz.partial_ratio(str(x).lower(), target_name.lower()) >= threshold
)

# Filter the DataFrame
filtered_emails = df_exploded[matches]
print(f"Total matching emails found: {len(filtered_emails)}")
filtered_emails[["from_email", "email_text_content"]].head()

**Example output (anonymized):**

```
Total matching emails found: 386

   from_email                       email_text_content
2  Sender, Example (Dept)           Below is a colleague's response to the inquiry...
4  Sender, Example (Dept)           I am working with the archive team on...
8  Sender, Example (Dept)           Welcome back! Attached is the updated plan...
```


## 📊 Step 8: Analyzing Email Volume Over Time

We perform a temporal analysis of email activity based on the `sent_email` field:

1. **Datetime Conversion** — Convert `sent_email` to datetime using `pd.to_datetime`.
2. **Filtering for a Specific Period** — Isolate a target month/year for focused analysis.
3. **Grouping by Month and Year** — Create a `year_month` column using `.dt.to_period("M")`.
4. **Summary Table** — Count emails per month.
5. **Visualization** — Horizontal bar chart of emails per month.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Convert 'sent_email' column to datetime
df_exploded["sent_email_parsed"] = pd.to_datetime(df_exploded["sent_email"], errors="coerce")

# Example: filter for a specific month/year
target_year = 2003
target_month = 7

emails_filtered = df_exploded[
    (df_exploded["sent_email_parsed"].dt.year == target_year) &
    (df_exploded["sent_email_parsed"].dt.month == target_month)
]

print(f"Emails in {target_month}/{target_year}: {len(emails_filtered)}")

In [ ]:
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Create Year-Month column
df_exploded["year_month"] = df_exploded["sent_email_parsed"].dt.to_period("M")

# Count emails per month
monthly_counts = df_exploded["year_month"].value_counts().sort_index()
monthly_counts_df = monthly_counts.reset_index()
monthly_counts_df.columns = ["Year_Month", "Email_Count"]

# Horizontal bar chart
plt.figure(figsize=(12, 8))
plt.barh(monthly_counts_df["Year_Month"].astype(str), monthly_counts_df["Email_Count"])
plt.xlabel("Number of Emails")
plt.ylabel("Year-Month")
plt.title("📬 Number of Emails Sent Per Month")
plt.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Bar chart for a specific sender over time (using fuzzy-matched results)
warnings.filterwarnings("ignore")

filtered_emails["sent_email_parsed"] = pd.to_datetime(filtered_emails["sent_email"], errors="coerce")
filtered_emails["year_month"] = filtered_emails["sent_email_parsed"].dt.to_period("M")

sender_monthly = filtered_emails["year_month"].value_counts().sort_index()
sender_monthly_df = sender_monthly.reset_index()
sender_monthly_df.columns = ["Year_Month", "Email_Count"]

plt.figure(figsize=(12, 6))
plt.bar(sender_monthly_df["Year_Month"].astype(str), sender_monthly_df["Email_Count"])
plt.xticks(rotation=45)
plt.xlabel("Year-Month")
plt.ylabel("Number of Emails")
plt.title("📧 Number of Emails from Target Sender Over Time")
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 🧠 Step 9: Simple Topic Modeling with Bar Chart Visualization

This code analyzes subject lines to discover main themes or topics being discussed.

1. **Cleans Subject Lines** — Removes punctuation, numbers, and stopwords.
2. **Identifies Common Topics** — Uses LDA to group subjects into distinct topic clusters.
3. **Assigns Each Subject a Topic** — Each email subject is labeled with its most likely topic.
4. **Visualizes Results** — Bar chart showing email volume per topic.


In [ ]:
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

stop_words = set(stopwords.words("english"))

df_exploded["subject_clean"] = df_exploded["subject_email"].apply(clean_text)
df_exploded["subject_clean"] = df_exploded["subject_clean"].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)

# Vectorize
vectorizer = CountVectorizer(max_df=0.9, min_df=2, stop_words="english")
doc_term_matrix = vectorizer.fit_transform(df_exploded["subject_clean"])

# LDA topic modeling
num_topics = 5
lda_model = LatentDirichletAllocation(n_components=num_topics, random_state=42)
lda_topics = lda_model.fit_transform(doc_term_matrix)

df_exploded["dominant_topic"] = np.argmax(lda_topics, axis=1)
topic_counts = df_exploded["dominant_topic"].value_counts().sort_index()

# Bar chart
plt.figure(figsize=(8, 5))
topic_counts.plot(kind="bar")
plt.title("Top Topics Found in Email Subjects")
plt.xlabel("Topic Number")
plt.ylabel("Number of Emails")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Display top keywords per topic
def display_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_indices = topic.argsort()[-top_n:][::-1]
        top_features = [feature_names[i] for i in top_features_indices]
        print(f"Topic #{topic_idx}: {', '.join(top_features)}")

feature_names = vectorizer.get_feature_names_out()
display_topics(lda_model, feature_names, top_n=10)

**Example topic keyword output:**

```
Topic #0: power, plant, project, question, gas, line, procurement, equipment, work, schedule
Topic #1: work, order, app, data, problems, expansion, system, request, maintenance
Topic #2: steam, gas, plant, info, maintenance, visit, controls, tunnel, service, repair
Topic #3: boiler, report, fuel, meeting, oil, gas, bid, change, plan, outage
Topic #4: ok, boiler, water, po, outage, status, notification, treatment, chemical
```


## 🧠 Step 10: Topic Modeling with LDA and Local LLM (Llama 3.2)

We used LDA to extract top keywords per topic, then passed those keywords to a local LLM (Llama 3.2 via Ollama) to generate human-readable topic titles.

**Process:**
- Preprocess subject lines (clean text, remove stopwords)
- Vectorize using CountVectorizer (bag-of-words)
- Apply LDA to uncover hidden topics
- Extract top 10 keywords per topic
- Pass keywords to Llama 3.2 to generate natural language topic titles

This converts abstract keyword clusters into intuitive, human-readable labels.


In [ ]:
import subprocess

# Path to Ollama executable
ollama_path = r"C:\Users\[USERNAME]\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"

def generate_topic_title(keywords):
    prompt = f"""
You are an expert at topic modeling and naming. Based on the following list of keywords,
provide a short descriptive title (5 words or fewer) for this topic.
Keywords: {', '.join(keywords)}
"""
    try:
        result = subprocess.run(
            [ollama_path, "run", model_name, "--", prompt],
            input=prompt,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace"
        )
        if result.returncode != 0:
            print("Error generating topic title:", result.stderr)
            return ""
        return result.stdout.strip()
    except Exception as e:
        print(f"An error occurred during topic title generation: {e}")
        return ""

# Generate titles for each topic
topic_titles = {}
for topic_idx, topic in enumerate(lda_model.components_):
    top_features_indices = topic.argsort()[-10:][::-1]
    top_keywords = [feature_names[i] for i in top_features_indices]
    topic_title = generate_topic_title(top_keywords)
    topic_titles[topic_idx] = {
        "keywords": top_keywords,
        "generated_title": topic_title
    }
    print(f"Topic #{topic_idx} → Title: {topic_title}")

**Example LLM-generated topic titles:**

```
Topic #0 → Title: "Power Plant Procurement Project"
Topic #1 → Title: "Work Order and System Management"
Topic #2 → Title: "Steam Plant Operations and Controls"
Topic #3 → Title: "Energy Contract and Fuel Planning"
Topic #4 → Title: "Boiler Water System Outage Response"
```


In [ ]:
# Plot bar chart with Llama-generated topic labels
topic_counts_df = topic_counts.reset_index()
topic_counts_df.columns = ["Topic", "Email_Count"]
topic_counts_df["Topic_Label"] = topic_counts_df["Topic"].apply(
    lambda x: topic_titles.get(x, {}).get("generated_title", f"Topic #{x}")
)

plt.figure(figsize=(10, 6))
plt.bar(topic_counts_df["Topic_Label"], topic_counts_df["Email_Count"], edgecolor='black')
plt.title("📬 Email Subject Topics (Named by Llama 3.2)")
plt.xlabel("Llama-Generated Topic Title")
plt.ylabel("Number of Emails")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---

## 🔐 Step 11: PII Detection with `presidio_analyzer`

The `presidio-analyzer` library (by Microsoft) is a powerful Python tool for detecting, analyzing, and classifying Personally Identifiable Information (PII) in unstructured text.

📖 Documentation: https://microsoft.github.io/presidio/analyzer/

### 📦 What is `AnalyzerEngine`?

The core component is the `AnalyzerEngine`. It scans text for sensitive information and labels it using structured entity types such as names, phone numbers, dates, and more.

**Entity types detected include:**
- Names, phone numbers, email addresses, dates
- Credit card numbers and bank account details
- IP addresses, locations, and national ID numbers
- URLs, domains, and medical terms


In [ ]:
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()

# Example text (synthetic - no real PII)
text = "My name is John Doe and my number is 312-555-1234. I was born on 01/01/1990."
results = analyzer.analyze(text=text, language="en")

for result in results:
    print(result)

**Example output:**
```
type: UK_NHS, start: 37, end: 49, score: 1.0
type: PERSON, start: 11, end: 19, score: 0.85
type: DATE_TIME, start: 65, end: 75, score: 0.85
type: PHONE_NUMBER, start: 37, end: 49, score: 0.75
```


### 1. Anonymize: Replace Detected PII with Placeholders

In [ ]:
from presidio_anonymizer import AnonymizerEngine

anonymizer = AnonymizerEngine()
anonymized_result = anonymizer.anonymize(text=text, analyzer_results=results)
print(anonymized_result.text)

**Output:** `My name is <PERSON> and my number is <UK_NHS>. I was born on <DATE_TIME>.`

### 2. Anonymize with Redaction (Blackout Style)

In [ ]:
from presidio_anonymizer.entities import OperatorConfig

operators = {"DEFAULT": OperatorConfig("redact")}
anonymized_result = anonymizer.anonymize(text=text, analyzer_results=results, operators=operators)
print(anonymized_result.text)

**Output:** `My name is  and my number is . I was born on .`

### 3. Blackout Style with Block Characters

In [ ]:
def custom_blackout(text, results):
    redacted_text = text
    offset = 0
    for result in sorted(results, key=lambda x: x.start):
        start, end = result.start + offset, result.end + offset
        length = end - start
        blackout = "█" * length
        redacted_text = redacted_text[:start] + blackout + redacted_text[end:]
        offset += len(blackout) - length
    return redacted_text

print(custom_blackout(text, results))

**Output:** `My name is ████████ and my number is ████████████. I was born on ██████████.`

### 4. Masking: Show Only Last 4 Digits of Phone Number

In [ ]:
import re

def custom_mask_phone(text, results):
    new_text = text
    offset = 0
    for r in results:
        if r.entity_type == "PHONE_NUMBER":
            original = text[r.start:r.end]
            digits_only = re.sub(r'\D', '', original)
            if len(digits_only) <= 4:
                continue
            masked_digits = "*" * (len(digits_only) - 4) + digits_only[-4:]
            masked_iter = iter(masked_digits)
            formatted_masked = ''.join(
                next(masked_iter) if c.isdigit() else c for c in original
            )
            start = r.start + offset
            end = r.end + offset
            new_text = new_text[:start] + formatted_masked + new_text[end:]
            offset += len(formatted_masked) - (end - start)
    return new_text

print(custom_mask_phone(text, results))

**Output:** `My name is John and my number is ***-***-1234.`

### 5. Hashing: Convert Names to Anonymized Hash Values

In [ ]:
operators = {
    "PERSON": OperatorConfig("hash"),
    "EMAIL_ADDRESS": OperatorConfig("redact"),
    "CREDIT_CARD": OperatorConfig("redact"),
    "US_SSN": OperatorConfig("redact")
}

text_multi = "My name is Alex Sample and I work with Jordan Test. Contact me at alex@example.com. SSN: 123-45-6789."
results_multi = analyzer.analyze(text=text_multi, language="en")
anonymized_result = anonymizer.anonymize(text=text_multi, analyzer_results=results_multi, operators=operators)
print(anonymized_result.text)

**Why use hashing?**
- Protects identity while enabling duplicate tracking
- Consistent (same input → same hash every time)
- Irreversible (cannot recover original value)
- Useful for anonymization while still analyzing communication patterns


## 🔐 Step 12: Detecting and Masking PII with `presidio-analyzer` at Scale

This section applies PII detection across the full email dataset. The function `clean_text_with_presidio()` scans each email, replaces detected entities with numbered placeholders (e.g., `<PERSON1>`, `<EMAIL_ADDRESS1>`), and stores the mapping dictionaries for reference.

**What it detects:**
- `PERSON` → stored in `names` column
- `PHONE_NUMBER` → stored in `phone_numbers` column
- `EMAIL_ADDRESS` → stored in `email_addresses` column
- `DATE_TIME` → stored in `dates` column
- `US_SOCIAL_SECURITY_NUMBER` → stored in `ssn` column


In [ ]:
import os
import time
import pandas as pd
from presidio_analyzer import AnalyzerEngine

# Paths
input_csv = r"data/emails_parsed_cleaned.csv"
output_csv = r"data/emails_parsed_cleaned_PII_Extracted_v3.csv"

def clean_text_with_presidio(text):
    if not isinstance(text, str):
        text = ""
    
    analyzer = AnalyzerEngine()
    results = analyzer.analyze(text=text, language='en')
    
    pii_mapping = {
        "PERSON": "names",
        "PHONE_NUMBER": "phone_numbers",
        "EMAIL_ADDRESS": "email_addresses",
        "DATE_TIME": "dates",
        "US_SOCIAL_SECURITY_NUMBER": "ssn"
    }
    
    mapping_dicts = {v: {} for v in pii_mapping.values()}
    counters = {entity: 1 for entity in pii_mapping}
    
    filtered_results = [res for res in results if res.entity_type in pii_mapping]
    filtered_results.sort(key=lambda r: r.start, reverse=True)
    
    cleaned_text = text
    for res in filtered_results:
        entity = res.entity_type
        col_name = pii_mapping[entity]
        placeholder = f"<{entity}{counters[entity]}>"
        counters[entity] += 1
        mapping_dicts[col_name][placeholder] = text[res.start:res.end]
        cleaned_text = cleaned_text[:res.start] + placeholder + cleaned_text[res.end:]
    
    final_result = {"cleaned_text": cleaned_text}
    final_result.update(mapping_dicts)
    return final_result

# Load data
# df_exploded = pd.read_csv(input_csv, low_memory=False)

# Batch processing with resume support
# batch_size = 10
# for i in range(start_index, len(df_exploded), batch_size):
#     batch = df_exploded.iloc[i:i+batch_size].copy()
#     for index, row in batch.iterrows():
#         extraction = clean_text_with_presidio(row["email_text_content"])
#         for key in new_columns:
#             batch.at[index, key] = extraction.get(key, "") if key == "cleaned_text" else str(extraction.get(key, {}))
#     batch.to_csv(output_csv, index=False, mode='a', header=(i == 0))

print("PII extraction function defined. Uncomment batch loop to run on full dataset.")

### Sample PII Extraction Output (Anonymized Example)

Below is representative output showing the cleaned text and extracted PII mappings. All real names and content from the original archive have been replaced with generic placeholders.

In [ ]:
# Load the PII-extracted dataset
df = pd.read_csv(r"data/emails_parsed_cleaned_PII_Extracted_v3.csv")
print(f"Dataset shape: {df.shape}")
df.head(3)

In [ ]:
# Display cleaned text and PII fields for first 5 records
for i in range(5):
    print("-" * 80)
    print(f"\n📄 Record #{i + 1}\n")
    print("🧼 CLEANED TEXT:")
    print(df.loc[i, "cleaned_text"])
    print("\n👤 NAMES:")
    print(df.loc[i, "names"])
    print("\n📧 EMAIL ADDRESSES:")
    print(df.loc[i, "email_addresses"])
    print("\n📅 DATES:")
    print(df.loc[i, "dates"])
    print("\n")

**Example output (all real names and email content replaced with anonymized values):**

```
────────────────────────────────────────────────────────────────────────────────
📄 Record #1

🧼 CLEANED TEXT:
Hi all,
The document management plan is in its final stages and will be ready for
<PERSON4>'s review when they return on <DATE_TIME7>. It outlines a more formal
procedure for keeping everyone updated.

Currently working on:
1) Mapping workflow processes and learning equipment terminology
2) Researching best document management practices
3) Inventorying records in all locations

Inventorying is being done with <PERSON3> and began <DATE_TIME6>. Work will
take place <DATE_TIME5> from 9-12 for <DATE_TIME4>.

<PERSON1>

👤 NAMES:
{'<PERSON1>': '[REDACTED]', '<PERSON2>': '[REDACTED]',
 '<PERSON3>': '[REDACTED]', '<PERSON4>': '[REDACTED]'}

📧 EMAIL ADDRESSES:
{}

📅 DATES:
{'<DATE_TIME1>': 'today', '<DATE_TIME2>': 'afternoon',
 '<DATE_TIME3>': 'yesterday', '<DATE_TIME4>': 'the next several weeks',
 '<DATE_TIME5>': 'each morning', '<DATE_TIME6>': '[DATE REDACTED]',
 '<DATE_TIME7>': '[DATE REDACTED]'}
────────────────────────────────────────────────────────────────────────────────
📄 Record #2

🧼 CLEANED TEXT:
By CC, all management staff is aware.

<PERSON1>

👤 NAMES:  {'<PERSON1>': '[REDACTED]'}
📧 EMAIL ADDRESSES: {}
📅 DATES: {}
```


## ⏱️ Estimated Processing Times for PII Extraction with Presidio

The following table shows estimated processing times based on observed batch performance.

| Batch Time (10 rows) | Avg Per Row | Time for 100,000 Rows | Time for 1,000,000 Rows |
|---|---|---|---|
| 10 seconds | 1.0 sec | ~27.8 hours | ~278 hours |
| 12 seconds | 1.2 sec | ~33.3 hours | ~333 hours |
| 15 seconds | 1.5 sec | ~41.7 hours | ~417 hours |
| 20 seconds | 2.0 sec | ~55.6 hours | ~556 hours |

**Notes:**
- Times assume sequential processing with batch size of 10
- Actual runtimes vary with system performance and email length
- Parallelization or persistent `AnalyzerEngine` usage could significantly reduce time


---

## 🧠 Step 13: Rewriting Email Text Using a Local LLM (Ollama / Llama 3.2)

This function sends each cleaned (PII-extracted) email to a local LLM via the Ollama CLI, which rewrites it as a concise, fully anonymized summary. Personal details are replaced with generic terms like `"the writer"` or `"a contact"`.

**Configuration:**
- `ollama_path` — Points to the installed Ollama executable
- `model_name` — Specifies the LLM (e.g., `"llama3.2"`)

**Function steps:**
1. Builds a structured prompt asking the model to summarize and anonymize
2. Sends the prompt via `subprocess.run()`
3. Returns the model's plain text response
4. Handles errors gracefully with try/except


In [ ]:
import subprocess
import os

ollama_path = r"C:\Users\[USERNAME]\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"

def rewrite_email_summary(email_text):
    prompt = f"""
Rewrite the following email as a concise summary that omits or generalizes all
sensitive or personal details. Replace names with "the writer" or "a contact",
remove specific dates, phone numbers, and locations.

Email text:
{email_text}
"""
    try:
        result = subprocess.run(
            [ollama_path, "run", model_name, "--", prompt],
            input=prompt,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace"
        )
        if result.returncode != 0:
            print("Error running summary prompt:", result.stderr)
            return ""
        return result.stdout.strip()
    except Exception as e:
        print(f"An error occurred during rewriting: {e}")
        return ""

print("rewrite_email_summary() function defined.")

**Example: Input vs. Output (synthetic test case)**

**Input (cleaned text with PII placeholders):**
```
The patient's name is <PERSON1>, their DOB is <DATE1>, their number is <PHONE1>,
and a contact has reached out regarding scheduling an appointment. The patient is
having pain on one side and needs to be evaluated. The left side has healed well.
Please email at <EMAIL1> and the address is <ADDRESS1>. <SSN1>
```

**Output (LLM-generated anonymized summary):**
```
A contact regarding a patient's appointment inquired about scheduling another
visit to evaluate symptoms on one side, while noting improvement on the other.
The contact can be reached via email and a physical location for correspondence.
```


## 📄 Step 14: Rewriting Email Summaries in Batches and Saving to CSV

This step applies the `rewrite_email_summary()` function across the full dataset in batches of 10 rows. It supports resuming from where it left off if interrupted.

**Steps:**
1. Load the PII-extracted CSV
2. Check for existing output to enable resume
3. Process in batches of 10 rows
4. Append results to output CSV after each batch
5. Track and print runtime per batch

**Output file:** `emails_parsed_rewritten_summaries.csv`  
Contains the original cleaned text plus a new `email_summaries` column with LLM-generated anonymous summaries.


In [ ]:
# import os, time, pandas as pd, subprocess
# input_csv  = r"data/emails_parsed_cleaned_PII_Extracted_v3.csv"
# output_csv = r"data/emails_parsed_rewritten_summaries.csv"
# df = pd.read_csv(input_csv)

# # Resume support
# start_index = pd.read_csv(output_csv).shape[0] if os.path.exists(output_csv) else 0
# print(f"Starting from row {start_index}...")

# batch_size = 10
# for i in range(start_index, len(df), batch_size):
#     start_time = time.time()
#     batch = df.iloc[i:i+batch_size].copy()
#     for index, row in batch.iterrows():
#         summary = rewrite_email_summary(row["cleaned_text"])
#         batch.at[index, "email_summaries"] = summary
#     if i == 0 and not os.path.exists(output_csv):
#         batch.to_csv(output_csv, index=False, mode='w', header=True)
#     else:
#         batch.to_csv(output_csv, index=False, mode='a', header=False)
#     print(f"✅ Rows {i}–{min(i+batch_size-1, len(df)-1)} in {time.time()-start_time:.1f}s")

print("Batch summarization loop defined. Uncomment to run on full dataset.")

### Sample Summary Output (Anonymized)

Below are representative examples of cleaned email text paired with LLM-generated summaries. All real names and identifying content from the original archive are replaced.

```
────────────────────────────────────────────────────────────────────────────────
📄 Record #1

🧼 CLEANED TEXT:
Hi all,
The document management plan is in its final stages and will be ready for
<PERSON4>'s viewing when they return on <DATE_TIME7>. [...]

📧 EMAIL SUMMARY:
A document management plan is nearing completion and will be ready for review
upon the recipient's return. The writer is currently mapping workflow processes,
researching best practices, and inventorying records. Inventory work is being
conducted collaboratively and scheduled to minimize disruption.

────────────────────────────────────────────────────────────────────────────────
📄 Record #2

🧼 CLEANED TEXT:
By CC, all management staff is aware.
<PERSON1>

📧 EMAIL SUMMARY:
The decision has been made, and all relevant parties are informed.

────────────────────────────────────────────────────────────────────────────────
📄 Record #3

🧼 CLEANED TEXT:
<PERSON6>,
Below is <PERSON5>'s response to my request to meet with them. They referred me
to <PERSON4>, but it looks good so far. Do I need to notify anyone else?
<PERSON1>

📧 EMAIL SUMMARY:
A meeting with a contact has been scheduled, and another individual has been
informed of the arrangement. The writer is unclear if additional parties need
to be notified about the upcoming event.

────────────────────────────────────────────────────────────────────────────────
📄 Record #4

🧼 CLEANED TEXT:
Sounds good! Glad you can support us in this manner. I am forwarding your email
to <PERSON1>, who is the point of contact for environmental compliance.

📧 EMAIL SUMMARY:
An individual has expressed support for an environmental compliance matter. Their
email will be forwarded to a designated point of contact within the organization.
```


## 📊 Step 15: Topic Modeling on LLM-Generated Summaries

We repeat the LDA + Llama topic modeling pipeline, this time running on the LLM-generated summaries rather than the raw subject lines. This produces cleaner, more semantically coherent topics because the summaries have already been anonymized and normalized.


In [ ]:
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt, subprocess, nltk, warnings
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)

df = pd.read_csv(r"data/emails_parsed_rewritten_summaries.csv")

def clean_text(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

stop_words = set(stopwords.words("english"))
df["summary_clean"] = df["email_summaries"].apply(clean_text)
df["summary_clean"] = df["summary_clean"].apply(
    lambda x: " ".join([w for w in x.split() if w not in stop_words])
)

vectorizer = CountVectorizer(max_df=0.9, min_df=2, stop_words="english")
doc_term_matrix = vectorizer.fit_transform(df["summary_clean"])

num_topics = 10
lda_model = LatentDirichletAllocation(n_components=num_topics, random_state=42)
lda_topics = lda_model.fit_transform(doc_term_matrix)
df["dominant_topic"] = np.argmax(lda_topics, axis=1)

topic_counts = df["dominant_topic"].value_counts().sort_index()
plt.figure(figsize=(8, 5))
topic_counts.plot(kind="bar")
plt.title("Top Topics Found in Email Summaries")
plt.xlabel("Topic Number"); plt.ylabel("Number of Emails")
plt.xticks(rotation=0); plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    top = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    print(f"Topic #{topic_idx}: {', '.join(top)}")

**Example topic output from summaries:**

```
Topic #0: contact, work, return, individual, status, regarding, date, request
Topic #1: plant, writer, operator, needs, information, contact, schedule, new
Topic #2: safety, records, summary, location, record, contact, retention, osha
Topic #3: available, access, project, contact, location, request, files, review
Topic #4: contact, employee, meeting, issue, writer, location, time, summary
Topic #5: writer, drawings, manuals, changes, contact, work, test, update
Topic #6: writer, contact, information, concise, details, sent, summary
Topic #7: employee, contact, resources, relations, related, writer, work
Topic #8: writer, time, meeting, period, status, benefits, employee
Topic #9: writer, time, location, request, individual, date, timecards

Topic #0 → "Record Access and Status Requests"
Topic #1 → "Plant Operation Management Guidelines"
Topic #2 → "OSHA Safety Record Keeping"
Topic #3 → "Project File Access Requests"
Topic #4 → "Employee Issue Resolution"
Topic #5 → "Technical Documentation Updates"
Topic #6 → "General Email Communication"
Topic #7 → "Employee Relations and HR"
Topic #8 → "Probationary Employee Onboarding"
Topic #9 → "Time and Location Scheduling"
```


---

## 📊 Pipeline Performance Summary

### ⏱️ Observed Runtimes

| Stage | Time (10 rows) | Avg/Row | 100K rows | 1M rows |
|---|---|---|---|---|
| PII Extraction (Presidio) | ~10–20 sec | ~1–2 sec | ~28–56 hrs | ~278–556 hrs |
| LLM Summarization (Ollama) | ~347 sec | ~34.78 sec | ~966 hrs | ~9,667 hrs |

**Current Bottleneck:** LLM summarization via local Ollama CLI is single-threaded and reloads the model each time.

---

## 🔚 Conclusion

The pipeline successfully structured and anonymized legacy email data for long-term archival and research access. PII extraction via `presidio-analyzer` was effective and reasonably efficient. LLM-based summarization via Ollama produced high-quality anonymized summaries but is too slow for large-scale processing without further optimization.

---

## ⚡ Optimization Opportunities

- **Parallelization** — Use `multiprocessing` or `joblib` to process rows concurrently
- **Persistent LLM Runtime** — Run Ollama in server mode to prevent model reload overhead
- **Prompt Simplification** — Shorten prompts to reduce token processing time
- **Cloud-Based Summarization** — Offload to faster APIs (e.g., Claude, OpenAI, Gemini)
- **Incremental Caching** — Track and skip already-processed rows using row ID hashing
- **GPU Acceleration** — Run Ollama on GPU hardware for dramatically faster inference
